
# GSM + GLEAM Closed-Loop Recovery (`gsm_recovery`)

This notebook adapts the closed-loop `newnucal` test to use a more realistic sky:

- diffuse emission from a Global Sky Model (GSM)
- compact emission from a GLEAM catalog
- smooth time/frequency gain perturbations
- radiometric complex visibility noise

The fit uses the existing `newnucal` infrastructure to recover sky, beam, and gain parameters and tracks the noise-weighted reduced chi-squared. The practical goal is to drive the fit to $\chi^2_\nu \approx 1$.

This notebook is written to be editable. The two most likely things you may need to adjust locally are:

1. the import path for the GSM package (`pygdsm` / `pygsm`)
2. the path and column names for the GLEAM catalog file


## 1. Imports and plotting setup

In [ ]:

import os
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import healpy as hp

from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u
from astropy.table import Table

jax.config.update("jax_enable_x64", False)

from newnucal import (
    HERAArray,
    BeamModel,
    ForwardModel,
    Calibrator,
    apply_gains,
    init_gain_params,
)
from newnucal.dpss import dpss_matrix, dpss_project
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})


## 2. Configuration

In [ ]:

# --- Array / observing setup ---
hexnum = 4
sep_m = 14.6

nfreq = 32
freqs = np.linspace(50e6, 225e6, nfreq)  # Hz
freq_mhz = freqs / 1e6
dnu_hz = float(np.median(np.diff(freqs)))

ntime = 8
integration_time_s = 7.5 * 60.0  # 7.5 minutes per integration

hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
times = t0 + np.arange(ntime) * integration_time_s * u.s

# --- Sky / beam resolution and spectral smoothness ---
sky_nside = 32
beam_nside = 16
beam_eta_max = 20e-9
sky_eta_max = 60e-9

# --- Sky model ingredients ---
ref_freq_hz = 151e6
gleam_catalog_path = Path("GLEAM_EGC_v2.fits")   # edit if needed
gleam_flux_col_candidates = ["int_flux_151", "S_151", "Fint151", "peak_flux_151"]
gleam_alpha_col_candidates = ["alpha", "sp_index", "SpectralIndex"]
max_gleam_sources = 5000

# --- Noise model ---
T_rx_K = 100.0
aperture_efficiency = 0.7
noise_seed = 1234

# --- Joint fit configuration ---
fit_kwargs = dict(
    sky_step_size=0.9,
    sky_anderson_history=3,
    sky_aa_start=1,
    beam_anderson_history=2,
    beam_aa_start=1,
    beam_aa_damping=0.5,
    beam_aa_ridge=1e-8,
    beam_aa_max_weight=10,
    solve_every={
        'gains':         10,   # hard cap: gains at least every 10 dirty steps
        'beam':           1,   # beam dirty steps enabled
        'sky_max':        5,   # force sky at least every 5 non-sky steps
        'beam_max':       5,   # force beam at least every 5 non-beam steps
        'beam_lbfgs_max': 60,  # force beam L-BFGS every 25 total steps
    },
    beam_step_size=0.5,
    verbose=True,
)

target_reduced_chi2 = 1.05
max_fit_rounds = 4


## 3. Helper functions for GSM, GLEAM, and radiometric noise

In [ ]:

k_B = 1.380649e-23
c = 299792458.0

def load_gsm_temperature_maps(freqs_hz, nside):
    gsm = None
    try:
        from pygdsm import GlobalSkyModel2016
        gsm = GlobalSkyModel2016(freq_unit="Hz")
    except Exception:
        pass
    if gsm is None:
        try:
            from pygdsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel2016
            gsm = GlobalSkyModel2016(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        try:
            from pygsm import GlobalSkyModel
            gsm = GlobalSkyModel(freq_unit="Hz")
        except Exception:
            pass
    if gsm is None:
        raise ImportError(
            "Could not import a GSM model. Install pygdsm/pygsm or edit "
            "load_gsm_temperature_maps() for your local GSM package."
        )

    # Galactic -> Equatorial (Celestial/ICRS-like) rotator
    rot_gc = hp.Rotator(coord=["G", "C"])
    maps = []
    for nu in freqs_hz:
        try:
            m_gal = gsm.generate(float(nu))
        except TypeError:
            # some GSM packages expect MHz
            m_gal = gsm.generate(float(nu) / 1e6)
        m_gal = np.asarray(m_gal, dtype=np.float64)
        # Rotate from Galactic to Equatorial before any regridding
        m_eq = rot_gc.rotate_map_pixel(m_gal)
        if hp.get_nside(m_eq) != nside:
            m_eq = hp.ud_grade(m_eq, nside_out=nside, power=0)
        maps.append(m_eq.astype(np.float32))
    return np.stack(maps, axis=1)  # (npix, nfreq)

def _pick_column(tab, candidates):
    for name in candidates:
        if name in tab.colnames:
            return name
    return None

def load_gleam_sources(path, max_sources=None):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"GLEAM catalog not found at {path}. Point the notebook at a local "
            "GLEAM FITS/VOTable file."
        )

    tab = Table.read(path)
    ra_col = _pick_column(tab, ["RAJ2000", "ra", "RA", "RA_deg"])
    dec_col = _pick_column(tab, ["DEJ2000", "dec", "DEC", "DEC_deg"])
    flux_col = _pick_column(tab, gleam_flux_col_candidates)
    alpha_col = _pick_column(tab, gleam_alpha_col_candidates)
    if ra_col is None or dec_col is None or flux_col is None:
        raise ValueError(
            "Could not identify RA/DEC/151-MHz flux columns in the GLEAM catalog. "
            "Edit the candidate column lists near the top of the notebook."
        )

    ra_deg = np.asarray(tab[ra_col], dtype=np.float64)
    dec_deg = np.asarray(tab[dec_col], dtype=np.float64)
    s151_jy = np.asarray(tab[flux_col], dtype=np.float64)

    if alpha_col is None:
        alpha = -0.8 * np.ones_like(s151_jy)
    else:
        alpha = np.asarray(tab[alpha_col], dtype=np.float64)
        alpha[~np.isfinite(alpha)] = -0.8

    good = np.isfinite(ra_deg) & np.isfinite(dec_deg) & np.isfinite(s151_jy) & (s151_jy > 0)
    ra_deg, dec_deg, s151_jy, alpha = ra_deg[good], dec_deg[good], s151_jy[good], alpha[good]

    if max_sources is not None and len(s151_jy) > max_sources:
        keep = np.argsort(s151_jy)[-max_sources:]
        ra_deg, dec_deg, s151_jy, alpha = ra_deg[keep], dec_deg[keep], s151_jy[keep], alpha[keep]

    return dict(ra_deg=ra_deg, dec_deg=dec_deg, s151_jy=s151_jy, alpha=alpha)

def gleam_flux_cube_to_healpix(freqs_hz, nside, gleam, ref_freq_hz=151e6):
    npix = hp.nside2npix(nside)
    ra = np.asarray(gleam["ra_deg"])
    dec = np.asarray(gleam["dec_deg"])
    s151 = np.asarray(gleam["s151_jy"])
    alpha = np.asarray(gleam["alpha"])

    theta = np.deg2rad(90.0 - dec)
    phi = np.deg2rad(ra)
    pix = hp.ang2pix(nside, theta, phi)

    cube = np.zeros((npix, len(freqs_hz)), dtype=np.float32)
    scale = (freqs_hz[None, :] / ref_freq_hz) ** alpha[:, None]
    src_flux = s151[:, None] * scale
    for si, px in enumerate(pix):
        cube[px] += src_flux[si].astype(np.float32)
    return cube

def kelvin_to_jy_per_pix(temp_K, freqs_hz, nside):
    omega_pix = 4.0 * np.pi / hp.nside2npix(nside)
    prefac = 2.0 * k_B * (freqs_hz**2) / c**2 / 1e-26  # Jy / sr / K
    return temp_K * prefac[None, :] * omega_pix

def build_gsm_plus_gleam_flux_cube(freqs_hz, nside, gleam_path, max_sources=None):
    gsm_temp_K = load_gsm_temperature_maps(freqs_hz, nside)
    gsm_jy = kelvin_to_jy_per_pix(gsm_temp_K, freqs_hz, nside)
    gleam = load_gleam_sources(gleam_path, max_sources=max_sources)
    gleam_jy = gleam_flux_cube_to_healpix(freqs_hz, nside, gleam)
    return gsm_temp_K, gsm_jy, gleam, gleam_jy, gsm_jy + gleam_jy

def airy_collecting_area(diameter_m, aperture_eff=0.7):
    return aperture_eff * np.pi * (diameter_m / 2.0) ** 2

def approximate_radiometric_sigma_jy(freqs_hz, gsm_temp_K, dish_diameter_m, dnu_hz, tint_s,
                                     T_rx_K=100.0, aperture_eff=0.7):
    T_sky_f = np.mean(gsm_temp_K, axis=0)
    T_sys_f = T_sky_f + T_rx_K
    A_eff = airy_collecting_area(dish_diameter_m, aperture_eff=aperture_eff)
    sefd_jy = 2.0 * k_B * T_sys_f / A_eff / 1e-26
    sigma = sefd_jy / np.sqrt(2.0 * dnu_hz * tint_s)
    return sigma.astype(np.float32), T_sys_f.astype(np.float32)

def complex_gaussian_noise(shape, sigma_f, rng):
    sigma = np.asarray(sigma_f, dtype=np.float32)[None, :, None]
    nre = rng.standard_normal(shape).astype(np.float32)
    nim = rng.standard_normal(shape).astype(np.float32)
    return sigma * (nre + 1j * nim) / np.sqrt(2.0)


## 4. Array, beam, times, and rotation matrices

In [ ]:

array = HERAArray.from_hex(hexnum=hexnum, sep=sep_m)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")

beam_model = BeamModel(nside=beam_nside, freqs=freqs, eta_max=beam_eta_max)
print(f"Beam DPSS modes: {beam_model.A_beam.shape[1]}")


## 5. Build the GSM + GLEAM sky cube and project to the sky DPSS basis

In [ ]:

gsm_temp_K, gsm_jy, gleam, gleam_jy, flux_true = build_gsm_plus_gleam_flux_cube(
    freqs,
    sky_nside,
    gleam_catalog_path,
    max_sources=max_gleam_sources,
)

print(f"GSM temperature cube shape: {gsm_temp_K.shape}")
print(f"GLEAM source count used:    {len(gleam['s151_jy'])}")
print(f"Combined sky flux cube:     {flux_true.shape}  [Jy / pixel]")

A_sky = dpss_matrix(freqs, sky_eta_max)
sky_coeffs_true = jnp.array(dpss_project(flux_true, A_sky), dtype=jnp.float32)

print(f"Sky DPSS modes:             {A_sky.shape[1]}")
print(f"sky_coeffs_true shape:      {sky_coeffs_true.shape}")


## 6. Simulate noiseless visibilities and inject smooth gains + radiometric noise

In [ ]:

fwd = ForwardModel(array, sky_nside, beam_model, freqs, eps=1e-5)
fwd.set_sky_dpss(A_sky)

print("Simulating noiseless visibilities (first call compiles JIT)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape}")
print(f"Mean |vis_true|: {float(jnp.abs(vis_true).mean()):.4e}")

t_norm = np.linspace(0, 1, ntime)[:, None]
f_norm = np.linspace(0, 1, nfreq)[None, :]

true_log_amp = (
    0.05 * np.cos(2 * np.pi * f_norm)
    + 0.02 * np.sin(2 * np.pi * t_norm)
).astype(np.float32)

true_phase = (
    0.15 * np.sin(2 * np.pi * f_norm)
    + 0.05 * np.cos(2 * np.pi * t_norm)
).astype(np.float32)

true_phi = np.zeros((ntime, 2, nfreq), dtype=np.float32)
true_phi[:, 0, :] = 1e-4 * np.cos(2 * np.pi * f_norm) + 3e-5 * np.sin(2 * np.pi * t_norm)
true_phi[:, 1, :] = 5e-5 * np.sin(2 * np.pi * f_norm) + 2e-5 * np.cos(2 * np.pi * t_norm)

vis_gain_only = apply_gains(
    vis_true,
    jnp.array(true_log_amp),
    jnp.array(true_phase),
    jnp.array(true_phi),
    jnp.array(array.bls, dtype=jnp.float32),
)

sigma_vis_f, Tsys_f = approximate_radiometric_sigma_jy(
    freqs,
    gsm_temp_K,
    dish_diameter_m=14.6,
    dnu_hz=dnu_hz,
    tint_s=integration_time_s,
    T_rx_K=T_rx_K,
    aperture_eff=aperture_efficiency,
)
rng = np.random.default_rng(noise_seed)
noise = complex_gaussian_noise(vis_gain_only.shape, sigma_vis_f, rng)

vis_data = vis_gain_only + jnp.array(noise)
noise_sigma = jnp.array(sigma_vis_f, dtype=jnp.float32)[None, :, None]

print(f"sigma_vis_f range [Jy]: {sigma_vis_f.min():.3e} -- {sigma_vis_f.max():.3e}")
print(f"Tsys range [K]:         {Tsys_f.min():.1f} -- {Tsys_f.max():.1f}")


## 7. Initial diagnostics and reference maps

In [ ]:

ref_ifreq = int(np.argmin(np.abs(freqs - ref_freq_hz)))

fig = plt.figure(figsize=(12, 9))
hp.mollview(gsm_jy[:, ref_ifreq], fig=fig, sub=(3, 1, 1), title="Diffuse GSM only [Jy/pix]", cmap="inferno")
hp.mollview(gleam_jy[:, ref_ifreq], fig=fig, sub=(3, 1, 2), title="GLEAM-only rasterized map [Jy/pix]", cmap="inferno")
hp.mollview(flux_true[:, ref_ifreq], fig=fig, sub=(3, 1, 3), title="GSM + GLEAM truth [Jy/pix]", cmap="inferno")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(freq_mhz, Tsys_f, label="Approx. Tsys")
plt.xlabel("Frequency [MHz]")
plt.ylabel("Temperature [K]")
plt.title("Approximate radiometric system temperature")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 8. Build the calibrator and define reduced chi-squared

In [ ]:

cal = Calibrator(
    array=array,
    beam_model=beam_model,
    sky_nside=sky_nside,
    sky_eta_max=sky_eta_max,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

beam_coeffs_true = jnp.array(beam_model.coeffs, dtype=jnp.float32)

def reduced_chi2(cal, params, sigma_vis):
    resid = cal.simulate(params, explicit_beam=True) - cal.data
    chi2 = jnp.sum(jnp.abs(resid) ** 2 / (sigma_vis ** 2))
    return float(chi2 / resid.size)

params_truth = {
    "sky_coeffs": sky_coeffs_true,
    "beam_coeffs": beam_coeffs_true,
    "log_amp": jnp.array(true_log_amp),
    "phase": jnp.array(true_phase),
    "phi": jnp.array(true_phi),
}

print("Truth reduced chi^2 (should be close to 1 with injected noise):")
print(reduced_chi2(cal, params_truth, noise_sigma))


## 9. Perturb the sky, beam, and gains to create the starting point

In [ ]:

rng2 = np.random.default_rng(99)
sky_coeffs_perturbed = sky_coeffs_true + 0.20 * jnp.array(
    rng2.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

beam_coeffs_perturbed = beam_coeffs_true + 0.20 * jnp.array(
    rng2.standard_normal(beam_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(beam_coeffs_true).mean())

gain0 = init_gain_params(ntime, nfreq)

params_joint = {
    "sky_coeffs": sky_coeffs_perturbed,
    "beam_coeffs": beam_coeffs_perturbed,
    **gain0,
}

print("Initial reduced chi^2:", reduced_chi2(cal, params_joint, noise_sigma))


## 10. Run the joint recovery until reduced chi-squared reaches ~1

In [ ]:

history = []
for fit_round in range(max_fit_rounds):
    print(f"\n=== fit round {fit_round + 1} / {max_fit_rounds} ===")
    params_joint, loss_fit = cal.fit_alternating_dirty(params_joint, n_iter=10, **fit_kwargs)
    chi2r = reduced_chi2(cal, params_joint, noise_sigma)
    history.append((fit_round, float(loss_fit), chi2r))
    print(f"round {fit_round + 1}: loss={loss_fit:.4e}   reduced chi^2={chi2r:.4f}")
    if chi2r <= target_reduced_chi2:
        print("Target reduced chi^2 reached.")
        break


## 11. Convergence history

In [ ]:

hist = np.array(history, dtype=float)
plt.figure(figsize=(7, 4))
plt.plot(hist[:, 0] + 1, hist[:, 1], "o-", label="Loss")
plt.yscale("log")
plt.xlabel("Fit round")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist[:, 0] + 1, hist[:, 2], "o-", label=r"Reduced $\chi^2$")
plt.axhline(1.0, color="k", ls="--", lw=1)
plt.axhline(target_reduced_chi2, color="r", ls=":", lw=1, label="Target")
plt.xlabel("Fit round")
plt.ylabel(r"Reduced $\chi^2$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 12. Compare recovered visibilities against the noisy data

In [ ]:

vis_fit = cal.simulate(params_joint, explicit_beam=True)
resid = vis_data - vis_fit

bl_indices = [0, array.nbls // 4, array.nbls // 2, array.nbls - 1]
t_show = 0
bls_j = jnp.array(array.bls, dtype=jnp.float32)

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(12, 2.8 * len(bl_indices)), sharex=True)
for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))
    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_data)[t_show, :, bi], "k-", lw=1.5, label="Data")
    ax.semilogy(freq_mhz, jnp.abs(resid)[t_show, :, bi], "r--", lw=1.2, label="Residual")
    ax.semilogy(freq_mhz, noise_sigma[0, :, 0], "b:", lw=1.0, label="Noise rms")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\n|V|")
    if row == 0:
        ax.legend(fontsize=8)
    if row == len(bl_indices) - 1:
        ax.set_xlabel("Frequency [MHz]")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_data)[t_show, :, bi], "k-", lw=1.5, label="Data")
    ax.plot(freq_mhz, jnp.angle(vis_fit)[t_show, :, bi], "r--", lw=1.2, label="Fit")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\narg(V)")
    if row == 0:
        ax.legend(fontsize=8)
    if row == len(bl_indices) - 1:
        ax.set_xlabel("Frequency [MHz]")
plt.tight_layout()
plt.show()


## 13. Sky and beam recovery diagnostics

In [ ]:

sky_rec = np.array(params_joint["sky_coeffs"] @ jnp.array(A_sky).T)
beam_rec = np.array(params_joint["beam_coeffs"] @ jnp.array(beam_model.A_beam).T)

fig = plt.figure(figsize=(12, 10))
hp.mollview(flux_true[:, ref_ifreq], fig=fig, sub=(3, 1, 1),
            title=f"True GSM+GLEAM sky ({freqs[ref_ifreq]/1e6:.0f} MHz)", cmap="inferno")
hp.mollview(sky_rec[:, ref_ifreq], fig=fig, sub=(3, 1, 2),
            title="Recovered sky", cmap="inferno")
hp.mollview(sky_rec[:, ref_ifreq] - flux_true[:, ref_ifreq], fig=fig, sub=(3, 1, 3),
            title="Recovered - true sky", cmap="RdBu_r")
plt.show()

fig = plt.figure(figsize=(12, 10))
beam_true_spec = np.array(beam_coeffs_true @ jnp.array(beam_model.A_beam).T)
hp.mollview(beam_true_spec[:, ref_ifreq], fig=fig, sub=(3, 1, 1),
            title=f"True beam ({freqs[ref_ifreq]/1e6:.0f} MHz)", cmap="inferno")
hp.mollview(beam_rec[:, ref_ifreq], fig=fig, sub=(3, 1, 2),
            title="Recovered beam", cmap="inferno")
hp.mollview(beam_rec[:, ref_ifreq] - beam_true_spec[:, ref_ifreq], fig=fig, sub=(3, 1, 3),
            title="Recovered - true beam", cmap="RdBu_r")
plt.show()
